# Convert a folder of Opta F24 exports into a corners table

Point `EVENTS_DIR` at a folder containing one `<date>...json` F24 export per match, and `MATCHES_CSV` at the companion match-list CSV (the standard Opta `matchInfo` export, one row per match -- see `wa_setpieces.convert.corners` docstring for the exact columns).

`build_corners_dataset` loads and matches **every** JSON file in the folder in one call -- there's no need to loop over files yourself. Requires `pip install "wa-setpieces[convert]"` (adds pyarrow, needed to write the parquet output at the end).

In [ ]:
from pathlib import Path

from wa_setpieces.convert.corners import build_corners_dataset, convert_to_parquet

EVENTS_DIR = Path("data/events")       # folder of <date>...json F24 exports
MATCHES_CSV = Path("data/matches.csv") # match-list CSV (matchInfo/... columns)
OUTPUT_PARQUET = Path("data/corners.parquet")

## See which JSON files are in the folder

Optional -- just to sanity-check the folder before converting. `build_corners_dataset` does its own `events_dir.glob(glob_pattern)` internally (default `"*.json"`), so this cell isn't required for the conversion itself.

In [ ]:
json_files = sorted(EVENTS_DIR.glob("*.json"))
print(f"{len(json_files)} JSON files found in {EVENTS_DIR}")
json_files[:5]

## Build the corners table

Each file is matched to a `MATCHES_CSV` row by (date parsed from the filename, contestant ID set) -- files with no matching row are skipped and reported below, not silently dropped.

In [ ]:
corners = build_corners_dataset(EVENTS_DIR, MATCHES_CSV)
print(f"{len(corners)} corner rows, {corners['match_id'].nunique()} matches")
corners.head()

## Write to parquet

`convert_to_parquet` does the same match + build step as above and writes the result straight to disk -- use it instead of the two cells above if you don't need the DataFrame in the notebook itself.

In [ ]:
convert_to_parquet(EVENTS_DIR, MATCHES_CSV, OUTPUT_PARQUET)